# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I use three simple first-pass models: logistic regression for a transparent linear baseline, a decision tree for a non-linear rule-based signal, and a random forest for a stronger ensemble. I keep the feature set conservative because the repository’s starter label is derived from the recent trend at the same export snapshot, so any feature built from the same trailing 90-day window would overlap the label. I therefore use only metadata and content-context features that are not derived from that overlap window.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

ROOT = Path.cwd()
if not (ROOT / 'scripts').exists():
    ROOT = Path(r'c:\Users\aryanks\Downloads\flyrank-ml-internship-starter-main')

features_path = ROOT / 'data' / 'processed' / 'refresh_feature_vector.csv'
baseline_path = ROOT / 'data' / 'processed' / 'baseline_refresh_queue.csv'

frame = pd.read_csv(features_path)
baseline_frame = pd.read_csv(baseline_path)
frame = frame.copy()

print('rows', len(frame))
print('positive_rate_full_frame', frame['is_declining_label'].mean())
print('unique_clients_full_frame', frame['client_id'].nunique())


rows 30000
positive_rate_full_frame 0.5420666666666667
unique_clients_full_frame 32


## 2. Split design

I use a grouped client-holdout split with `client_id` as the grouping key. This is the most honest split available in the starter data because the repo explicitly says `client_id` is for grouped validation and because rows from the same client can share hidden behavior that a random split would let the model memorize. I also keep the evaluation population aligned with the Week-4 baseline by using the same rows that exist in the prepared feature frame and the baseline queue. The starter dataset does not contain a true future-looking target or a fully leakage-free time-separated feature set, so the split is honest for the available data but it remains a proxy evaluation rather than a deployment-grade time-series test.


In [2]:
# Build the safe feature list from the starter data.
# Excluded features: any feature derived from the overlap window that could contain the label's information.
# Safe to use: content metadata, static content properties, and other non-overlapping context.

safe_numeric = [
    'search_volume',
    'competition',
    'cpc',
    'word_count',
    'char_count',
    'content_age_days',
    'days_since_last_update',
    'age_tier_order',
    'has_clicks',
    'has_ai_sessions',
    'measurable_opportunity',
]

safe_categorical = [
    'competition_level',
    'content_type',
    'main_intent',
    'age_tier',
    'freshness_tier',
    'word_count_tier',
    'char_count_tier',
]

# The starter data does not support a leakage-free time-separated feature set because the label is based on
# the same recent-vs-previous trend snapshot and the available features are already aggregated over the same export window.
excluded_features = [
    'impressions_90d',
    'clicks_90d',
    'pageviews_90d',
    'sessions_90d',
    'users_90d',
    'engaged_sessions_90d',
    'ai_sessions_90d',
    'scroll_events_90d',
    'days_with_impressions',
    'days_with_sessions',
    'impressions_last_30d',
    'clicks_last_30d',
    'sessions_last_30d',
    'impressions_prev_30d',
    'clicks_prev_30d',
    'sessions_prev_30d',
    'ctr',
    'avg_position',
    'engagement_rate',
    'scroll_rate',
    'ai_traffic_pct',
    'trend_pct',
    'trend_direction',
    'log_impressions_90d',
    'log_clicks_90d',
    'log_sessions_90d',
    'log_ai_sessions_90d',
]

available_features = [c for c in safe_numeric + safe_categorical if c in frame.columns]
print('safe_feature_list', available_features)
print('excluded_feature_count', len(excluded_features))

# Build a client-aware split on the prepared frame.
all_indices = np.arange(len(frame))
client_series = frame['client_id'].fillna('unknown').astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()

rng = np.random.default_rng(42)
shuffled_clients = rng.permutation(unique_clients)

# Hold out 20% of clients.
client_test_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:client_test_count])
test_mask = client_series.isin(test_clients).to_numpy()
train_indices = all_indices[~test_mask]
test_indices = all_indices[test_mask]

train_frame = frame.iloc[train_indices].copy()
test_frame = frame.iloc[test_indices].copy()

print('train_rows', len(train_frame))
print('test_rows', len(test_frame))
print('train_unique_clients', train_frame['client_id'].nunique())
print('test_unique_clients', test_frame['client_id'].nunique())
print('client_overlap', set(train_frame['client_id']) & set(test_frame['client_id']))
print('positive_rate_eval', test_frame['is_declining_label'].mean())


safe_feature_list ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update', 'age_tier_order', 'has_clicks', 'has_ai_sessions', 'measurable_opportunity', 'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier']
excluded_feature_count 27
train_rows 27675
test_rows 2325
train_unique_clients 26
test_unique_clients 6
client_overlap set()
positive_rate_eval 0.3909677419354839


## 3. Train + compare vs my baseline

The Week-4 baseline is evaluated on the same held-out client population as the learned models. The table below is produced from the actual held-out rows and uses the same metrics for every model: ROC-AUC, Average Precision, and Precision@50. The baseline is included as a comparison point, but it is still a heuristic score rather than a leakage-free oracle.


In [3]:
# Prepare train/test matrices with the conservative feature set.
# Use the same evaluation population as the Week-4 baseline where possible: rows present in the prepared frame.

X_train = train_frame[available_features]
X_test = test_frame[available_features]
y_train = train_frame['is_declining_label'].astype(int)
y_test = test_frame['is_declining_label'].astype(int)

numeric_features = [c for c in safe_numeric if c in X_train.columns]
categorical_features = [c for c in safe_categorical if c in X_train.columns]

preprocess = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_features),
    ],
    remainder='drop',
)

models = {
    'logistic_regression': Pipeline([
        ('preprocess', preprocess),
        ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)),
    ]),
    'decision_tree': Pipeline([
        ('preprocess', preprocess),
        ('model', DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, class_weight='balanced', random_state=42)),
    ]),
    'random_forest': Pipeline([
        ('preprocess', preprocess),
        ('model', RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=25, class_weight='balanced_subsample', n_jobs=-1, random_state=42)),
    ]),
}

# Baseline scores from Week 4 on the same evaluation population.
baseline_lookup = baseline_frame.set_index('content_id')['baseline_refresh_score']
base_scores = test_frame['content_id'].map(baseline_lookup).fillna(0).to_numpy()

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)
    rank_frame = pd.DataFrame({'y': y_test.to_numpy(), 'score': prob})
    top = rank_frame.sort_values('score', ascending=False).head(50)
    results.append({
        'model': name,
        'roc_auc': roc_auc_score(y_test, prob),
        'average_precision': average_precision_score(y_test, prob),
        'precision_at_50': float(top['y'].mean()) if len(top) else 0.0,
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
    })

base_prob = base_scores
base_pred = (base_prob >= 0.5).astype(int)
base_rank_frame = pd.DataFrame({'y': y_test.to_numpy(), 'score': base_prob})
base_top = base_rank_frame.sort_values('score', ascending=False).head(50)
results.append({
    'model': 'baseline_rules',
    'roc_auc': roc_auc_score(y_test, base_prob),
    'average_precision': average_precision_score(y_test, base_prob),
    'precision_at_50': float(base_top['y'].mean()) if len(base_top) else 0.0,
    'precision': precision_score(y_test, base_pred, zero_division=0),
    'recall': recall_score(y_test, base_pred, zero_division=0),
    'f1': f1_score(y_test, base_pred, zero_division=0),
})

results_df = pd.DataFrame(results)
results_df = results_df[['model', 'roc_auc', 'average_precision', 'precision_at_50', 'precision', 'recall', 'f1']].copy()
results_df


C:\Users\aryanks\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,model,roc_auc,average_precision,precision_at_50,precision,recall,f1
0,logistic_regression,0.675171,0.567883,0.78,0.549330,0.496150,0.521387
1,decision_tree,0.624484,0.512973,0.60,0.558614,0.602860,0.579894
2,random_forest,0.667588,0.520885,0.56,0.554745,0.585259,0.569593
3,baseline_rules,0.626892,0.467607,0.24,0.498551,0.189219,0.274322


## 4. Errors and interpretation

The error analysis below is based on the actual held-out predictions. I inspect false positives and false negatives directly so the discussion stays tied to the observed results rather than to a generic interpretation.


In [4]:
# Error analysis on the best-performing model.
# Keep the summary grounded in the computed predictions and the actual held-out rows.

best_model_name = results_df.sort_values(['average_precision', 'roc_auc', 'precision_at_50'], ascending=False).iloc[0]['model']
if best_model_name == 'baseline_rules':
    best_prob = base_prob
    best_pred = base_pred
else:
    best_model = models[best_model_name]
    best_model.fit(X_train, y_train)
    best_prob = best_model.predict_proba(X_test)[:, 1]
    best_pred = (best_prob >= 0.5).astype(int)

error_frame = test_frame[['content_id', 'client_id', 'is_declining_label']].copy()
error_frame['score'] = best_prob
error_frame['pred'] = best_pred
error_frame['correct'] = (error_frame['pred'] == error_frame['is_declining_label']).astype(int)
error_frame['error_type'] = np.where(
    (error_frame['pred'] == 1) & (error_frame['is_declining_label'] == 0),
    'false_positive',
    np.where(
        (error_frame['pred'] == 0) & (error_frame['is_declining_label'] == 1),
        'false_negative',
        'correct',
    ),
)

false_positives = error_frame[error_frame['error_type'] == 'false_positive']
false_negatives = error_frame[error_frame['error_type'] == 'false_negative']

print('best_model', best_model_name)
print('false_positive_count', len(false_positives))
print('false_negative_count', len(false_negatives))
print('false_positive_preview')
print(false_positives[['content_id', 'client_id', 'is_declining_label', 'score']].head(10).to_string(index=False))
print('false_negative_preview')
print(false_negatives[['content_id', 'client_id', 'is_declining_label', 'score']].head(10).to_string(index=False))

# Permutation importance for the best model, where appropriate.
if best_model_name != 'baseline_rules':
    perm = permutation_importance(
        best_model.named_steps['model'],
        best_model.named_steps['preprocess'].transform(X_test),
        y_test,
        n_repeats=10,
        random_state=42,
        scoring='average_precision',
    )
    feature_names = best_model.named_steps['preprocess'].get_feature_names_out()
    perm_frame = pd.DataFrame({'feature': feature_names, 'importance_mean': perm.importances_mean})
    perm_frame = perm_frame.sort_values('importance_mean', ascending=False).head(10)
    print('permutation_importance')
    print(perm_frame.to_string(index=False))
else:
    print('permutation_importance_not_applicable_for_baseline')


C:\Users\aryanks\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


best_model logistic_regression
false_positive_count 370
false_negative_count 458
false_positive_preview
          content_id         client_id  is_declining_label    score
content_95d488a56079 client_f74efabef1                   0 0.567208
content_d7cbd76b788d client_f74efabef1                   0 0.595006
content_c3e86d4031b6 client_f74efabef1                   0 0.661356
content_7dff534db3ae client_f74efabef1                   0 0.614915
content_f4e177f6d346 client_f74efabef1                   0 0.561600
content_29102284b855 client_f74efabef1                   0 0.587016
content_218ca439f951 client_f74efabef1                   0 0.652787
content_5920115cc1ad client_f74efabef1                   0 0.673705
content_36a91ab85be8 client_f74efabef1                   0 0.635823
content_e444c00065bd client_d4735e3a26                   0 0.565730
false_negative_preview
          content_id         client_id  is_declining_label    score
content_326fa2fa449f client_98a3ab7c34                   

permutation_importance
                         feature  importance_mean
                 num__word_count         0.082398
                 num__char_count         0.064482
           num__content_age_days         0.046865
     num__measurable_opportunity         0.041010
                 num__has_clicks         0.024505
  cat__main_intent_informational         0.013384
cat__content_type_feedly article         0.008516
           cat__age_tier_181-365         0.007695
  cat__competition_level_unknown         0.004660
        cat__main_intent_unknown         0.003886


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] I documented the leakage finding, the excluded features, the honest validation split, and the remaining limitations.
